## Load_Cutshort_Feed

| Metadata | Detail |
|-----------------------|----------------------|
| **Created By**        | Yateesh Chandra      |
| **Business Logic By** | Yateesh Chandra      |
| **Load Strategy**     | Append               |
| **Source**            | Scraping Cutshort             |
| **Target**            | jobsintel.bronze.raw_cutshort_jobs |

---

### History

| Date         | Modified By      | Change Log             |
|--------------|------------------|------------------------|
| June 18th 2026| Yateesh Chandra  | Created Initial Version|

In [0]:
import requests
import json
from bs4 import BeautifulSoup
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Define the URL
URL = "https://cutshort.io/jobs/python-jobs"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/125.0.0.0 Safari/537.36"
    )
}

# Define the Empty List 
payload_list = []
job_list = []

In [0]:
def fetch_cutshort_job_list(url):
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()

    # Extract the Reponse
    soup = BeautifulSoup(response.text, "html.parser")

    for card in soup.select("div.sc-8f06d440-0.ciLipX"):
        title = card.select_one("a.dIKnux")
        href = title.get("href", "") if title else ""
        job_list.append(href)
    return job_list

In [0]:
jobs = fetch_cutshort_job_list(URL)

In [0]:
def get_cutshort_job(jobs):
    for url in jobs:
        if url :
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()

            # Extract the Reponse
            soup = BeautifulSoup(response.text, "html.parser")

            title = soup.select_one("h1.bMdtwg")
            company = soup.select_one(".cfBBKA a") if soup.select_one(".cfBBKA a") else soup.select_one(".fSEpsP a")
            company_url = soup.select_one(".cfBBKA a") if soup.select_one(".cfBBKA a") else soup.select_one(".fSEpsP a")
            job_url = url
            job_id = url.rsplit("-", 1)[-1]
            details = [
                    s.get_text(strip=True)
                    for s in soup.select("div.hZbKiR")
                ]
            experience = details[0] if details else None
            salary = details[1] if details else None
            location = details[2] if details else None
            skills = [
                    s.get_text(strip=True)
                    for s in soup.select("div.sc-c52be3c1-15 span")
                ]
            job_desc = soup.select_one(".sc-54ec2143-2")

            payload = {
                "job_id": job_id if job_id else None,
                "title": title.get_text(strip=True) if title else None,
                "job_url": job_url if job_url else None,
                "company_name": company.get_text(strip=True) if company else None,
                "company_url": company_url.get("href") if company else None,
                "location": location if location else None,
                "experience": experience if experience else None,
                "salary": salary if salary else None,
                "job_desc" : job_desc.get_text(strip = True, separator = "\n") if job_desc else None,
                "skills" : ", ".join(skills) if skills else None
            }
            payload_list.append(json.dumps(payload, ensure_ascii=False))
    return payload_list

In [0]:
# Extract 50 jobs
val = get_cutshort_job(job_list)

In [0]:
cutshort_df = spark.createDataFrame(val,schema = ['PAYLOAD']) \
            .withColumn("BD_CREATE_DT_TM", current_timestamp()) \
            .withColumn("BD_UPDATE_DT_TM", current_timestamp())

In [0]:
refined_cutshort_df = cutshort_df.filter("payload:title is not null")

In [0]:
refined_cutshort_df.write.mode("append").saveAsTable("jobsintel.bronze.raw_cutshort_jobs")